# Tarstrade — Free Finetune (Kaggle T4 x2 = 30h/week)
Clone → QLoRA on free T4 with Unsloth. Works with `Qwen2.5-0.5B` (tiny, Android-runnable) or any HF model. No API bill, ever. Local + Kaggle dual path.
**Steps:** 1) Verify GPU 2) Install 3) Prepare data 4) QLoRA 5) Validate via Tarstrade gate 6) Export LoRA (<100MB)

In [ ]:
# Cell 1 — Verify GPU (Kaggle: T4 x2, 30h/week)
!nvidia-smi
!python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"

In [ ]:
# Cell 2 — Install (Unsloth = 2x faster, 70% less VRAM than vanilla)
!pip install -q unsloth transformers peft datasets accelerate trl bitsandbytes
# For local Windows: pip install torch --index-url https://download.pytorch.org/whl/cpu  # then same pip line without bitsandbytes

In [ ]:
# Cell 3a — Option A: Tarstrade native pipeline (LightGBM, CPU, no GPU needed — SKIP if no data)
# data/ is gitignored — uncomment below ONLY if you uploaded CSVs as Kaggle Dataset or have them locally
# !git clone https://github.com/Henoch4/Tars.git tarstrade 2>&1 | tail -n 2
# !ls tarstrade/data | head
# import sys; sys.path.insert(0, 'tarstrade')
# !python -m tarstrade.ml.pipeline --symbols BTC  # smoke: 2-4 min on CPU, no GPU
# >>> QLoRA path starts at Cell 3b below — no CSV data needed <<<

In [ ]:
# Cell 3b — Load REAL training JSONL (built by scripts/build_lora_dataset.py)
# Requires: upload data/carry/lora_train.jsonl as a Kaggle Input (Add Input -> Dataset)
# OR copy it into the notebook working dir. It is 4185 rows, balanced 33% yes,
# from the real 2024-26 carry dataset (walk-forward, leakage-safe).
#
# NOTE: search is RECURSIVE over /kaggle/input — Kaggle mounts datasets deeply
# (e.g. /kaggle/input/<owner>/<slug>/lora_train.jsonl or .../datasets/...), so a
# single-level glob fails silently. If not found, it prints the whole input tree
# so you can see exactly where the file sits.
import os, glob, shutil, pathlib

# First, show the input tree so a miss is diagnosable (template idiom).
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

candidates = list(glob.glob('/kaggle/input/**/lora_train.jsonl', recursive=True))
candidates += [str(p) for p in pathlib.Path('.').glob('lora_train.jsonl')]
candidates += [str(p) for p in pathlib.Path('.').glob('train.jsonl')]
candidates = list(dict.fromkeys(candidates))  # dedupe, keep order
print('candidates:', candidates)
if not candidates:
    raise SystemExit('lora_train.jsonl not found — upload it as an Input (Add Input -> Dataset), then re-run from Cell 3b')
src = candidates[0]
shutil.copy(src, 'train.jsonl')
n = sum(1 for _ in open('train.jsonl'))
print(f'loaded {src} -> train.jsonl ({n} rows)')
print(open('train.jsonl').readline())


In [ ]:
# Cell 4 — QLoRA finetune (4-bit, fits free T4 — Qwen2.5-0.5B is Android-runnable)
# FIX (must be FIRST): Kaggle T4 x2 spreads model across cuda:0+cuda:1 and crashes
# on the embedding index_select. Pin to one GPU before torch is imported.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Real data: 4185 balanced rows (~1400 yes / ~2790 no).
# Effective batch = per_device 2 x grad-accum 4 = 8.
# 4185 / 8 = ~523 steps/epoch -> run 2 epochs to cover the set ~1.5x.
N_STEPS = 1000
dataset = load_dataset("json", data_files="train.jsonl", split="train")
print('epochs at N_STEPS:', round(N_STEPS / (dataset.num_rows / 8), 2))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",  # 4-bit pre-quantized, <1GB
    max_seq_length = 2048,
    dtype = None, load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(model,
    r = 16, lora_alpha = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout = 0, bias = "none", use_gradient_checkpointing = True,
)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset, dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, max_steps=N_STEPS, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit", output_dir="outputs", save_steps=500,
    ),
)
trainer.train()
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA saved to lora_model/ (<100MB)")
!ls -lh lora_model | head

In [ ]:
# Cell 5 — Quick eval (matches the training question format)
for prompt in [
    "Funding 0.0003 basis 2.1 vol 0.0184 ret 0.021 7d_mean 0.0002 z 1.2 -> will 7d carry clear costs?",
    "Funding -0.0001 basis -8.5 vol 0.0073 ret 0.0142 7d_mean -0 z -1.6 -> will 7d carry clear costs?",
]:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=8, temperature=0.0, do_sample=False)
    print(prompt, "->", tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip())

In [ ]:
# Cell 6 — Export + validate via Tarstrade gate (no GPU needed)
# Merge for local inference (qvac / ollama) — still <1GB
model.save_pretrained_merged("qwen-custom-merged", tokenizer, save_method="merged_16bit")
print("Merged at qwen-custom-merged/")
# Validate: run your walk-forward gate locally or here
# !python tarstrade/ml/pipeline.py --symbols BTC,ETH,SOL,BNB  # then check reports/ml-v2-*.md Calmar>=1.0 PBO<=0.5

### Next — iterate weekly (RETRAINING_POLICY.md)
- Weekly retrain same features/labels on rolling 2y (168 new bars/week) — shadow 2 weeks before live weight.
- Push LoRA to Hub for free hosting: `huggingface-cli login` → `model.push_to_hub("Henoch/qwen-custom-nigeria")`.
- Run locally offline: `pip install ollama && ollama run qwen-custom-merged` or `qvac serve openai` at 127.0.0.1:11434.
- Cost: $0 marginal — Kaggle 30h + Hub + local CPU (LightGBM pipeline also CPU-only, 2-4 min/symbol).